## Imports

In [67]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [68]:
import numpy as np

In [69]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report,
    multilabel_confusion_matrix
)

from sklearn import preprocessing

In [70]:
from ucimlrepo import fetch_ucirepo

# Wine

Se observa que con regresiones logisticas los resultados son suficientemente competitivos (https://archive.ics.uci.edu/dataset/109/wine), lo cual explica porque con una o dos reglas basta para obtener un buen resultado.

## Data

In [71]:
wine = fetch_ucirepo(id=109)

X = wine.data.features
y = wine.data.targets

In [72]:
wine.variables

,name,role,type,demographic,description,units,missing_values
0,class,Target,Categorical,None,None,None,no
1,Alcohol,Feature,Continuous,None,None,None,no
2,Malicacid,Feature,Continuous,None,None,None,no
3,Ash,Feature,Continuous,None,None,None,no
4,Alcalinity_of_ash,Feature,Continuous,None,None,None,no
5,Magnesium,Feature,Integer,None,None,None,no
6,Total_phenols,Feature,Continuous,None,None,None,no
7,Flavanoids,Feature,Continuous,None,None,None,no
8,Nonflavanoid_phenols,Feature,Continuous,None,None,None,no
9,Proanthocyanins,Feature,Continuous,None,None,None,no


In [73]:
X

,Alcohol,Malicacid,Ash,Alcalinity_of_ash,Magnesium,Total_phenols,Flavanoids,Nonflavanoid_phenols,Proanthocyanins,Color_intensity,Hue,0D280_0D315_of_diluted_wines,Proline
0,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735
...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740
174,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750
175,13.27,4.28,2.26,20.0,120,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835
176,13.17,2.59,2.37,20.0,120,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840


In [74]:
y

,class
0,1
1,1
2,1
3,1
4,1
...,...
173,3
174,3
175,3
176,3


In [75]:
le = preprocessing.LabelEncoder()
y.loc[:, 'class'] = le.fit_transform(y['class'])

In [76]:
y = y.astype('int64')
y

,class
0,0
1,0
2,0
3,0
4,0
...,...
173,2
174,2
175,2
176,2


In [77]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, stratify=y_train)

In [78]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print("Training Data")
print(unique_classes)
print(counts)

Training Data
[0 1 2]
[33 40 26]


In [79]:
unique_classes, counts = np.unique(y_val.values, return_counts=True)
print("Validation Data")
print(unique_classes)
print(counts)

Validation Data
[0 1 2]
[ 8 10  7]


In [80]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print("Testing Data")
print(unique_classes)
print(counts)

Testing Data
[0 1 2]
[18 21 15]


In [81]:
scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)

x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

In [82]:
y_train.values.dtype

dtype('int64')

In [83]:
x_train = torch.from_numpy(x_train)
x_val = torch.from_numpy(x_val)
x_test = torch.from_numpy(x_test)

y_train = torch.from_numpy(y_train.values).squeeze()
y_val = torch.from_numpy(y_val.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

## Model & Algorithms

### DataLoaders

In [84]:
train_loader = data.DataLoader(
    data.TensorDataset(x_train, y_train), 
    batch_size = 8, 
    shuffle = True)
x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

In [85]:
val_loader = data.DataLoader(
    data.TensorDataset(x_val, y_val), 
    batch_size = 8, 
    shuffle = True)
x_val = val_loader.dataset.tensors[0]
y_val = val_loader.dataset.tensors[1]

### ANFIS

In [86]:
features = wine.variables["name"][:13].tolist()
features

['class',
 'Alcohol',
 'Malicacid',
 'Ash',
 'Alcalinity_of_ash',
 'Magnesium',
 'Total_phenols',
 'Flavanoids',
 'Nonflavanoid_phenols',
 'Proanthocyanins',
 'Color_intensity',
 'Hue',
 '0D280_0D315_of_diluted_wines']

In [87]:
model = nft.rule_reduced_ANFIS(
    input_size = x_train.shape[1],
    num_mfs = 1, # 8
    outputs = 3,
    default_rule=True,
    membership_function=nft.Gaussian_MF,
    output_type='softmax',
    features=features,
    dtype=torch.float64
)

#model.init_premises(x_train)

In [88]:
with torch.no_grad():
    pred = model(x_train)

nn.functional.cross_entropy(pred, y_train)

tensor(1.0986, dtype=torch.float64)

In [89]:
"""
model.init_consequents(x_train, y_train, driver="gelsd", ridge_lambda=1e-3)

with torch.no_grad():
    pred = model(x_train)

nn.functional.cross_entropy(pred, y_train)
"""

'\nmodel.init_consequents(x_train, y_train, driver="gelsd", ridge_lambda=1e-3)\n\nwith torch.no_grad():\n    pred = model(x_train)\n\nnn.functional.cross_entropy(pred, y_train)\n'

In [90]:
print(model.get_rules_structure().to_string())

        premises                                                                                                                                                                                                                                                                                                                  output 1 consequents                                                                                                                                                                                          output 2 consequents                                                                                                                                                                                          output 3 consequents                                                                                                                                                                                        
           class             Alcohol           Malicacid     

### Learning Algorithm

In [91]:
#loss_fn = nn.functional.binary_cross_entropy
loss_fn = nn.functional.cross_entropy
epochs = 200
#optimizer = torch.optim.Adam
#params = {'lr': 0.001}
optimizer = torch.optim.AdamW
params = {'lr': 0.001, 'weight_decay': 0.01}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = nft.EarlyStopping(
    patience=30, 
    #delta=0.0001
)

In [92]:
trainer = nft.Basic_optimizer_training_algorithm(
    epochs=epochs,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

In [93]:
"""
trainer(model, train_loader, val_loader, verbose=True)

pred = model.predict(x_test)

acc = accuracy_score(y_test, pred, )
prec = precision_score(y_test, pred, average='weighted', zero_division=0)
recall = recall_score(y_test, pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
class_rep = classification_report(y_test, pred)

print("\n ------ Evaluation ------")
print("\n")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)
"""

'\ntrainer(model, train_loader, val_loader, verbose=True)\n\npred = model.predict(x_test)\n\nacc = accuracy_score(y_test, pred, )\nprec = precision_score(y_test, pred, average=\'weighted\', zero_division=0)\nrecall = recall_score(y_test, pred, average=\'weighted\', zero_division=0)\nf1 = f1_score(y_test, pred, average=\'weighted\', zero_division=0)\nconf_matrix = confusion_matrix(y_test, pred)\nclass_rep = classification_report(y_test, pred)\n\nprint("\n ------ Evaluation ------")\nprint("\n")\nprint("Accuracy:", acc)\nprint("Precision:", prec)\nprint("Recall:", recall)\nprint("f1 score:", f1, "\n")\n\nprint("Confusion Matrix:")\nprint(conf_matrix, "\n")\n\nprint("Classification Report:")\nprint(class_rep)\n'

### SONFIS

In [94]:
Ngrow = 30
dGrow = 0.6
Nsplit = 60
eSplit = 0.3
Nvanish = 5
lVanish = 3

max_iterations = 100

anfis_trainer = trainer

sonfis_early_stopping = nft.EarlyStopping(patience=20)
last_training_iteration = False

lse_for_new_consequents = True
lse_for_new_consequents_lambda = 1e-2

In [95]:
sonfis = nft.SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    early_stopping=sonfis_early_stopping,
    lse_for_new_consequents=lse_for_new_consequents,
    lse_for_new_consequents_lambda=lse_for_new_consequents_lambda,
    last_training_iteration=last_training_iteration
)

## Training

In [96]:
%time sonfis(model, train_loader, val_loader, verbose=True)

ITERATION:   0/100

STARTING STATE:
   premises                                                                                                                                                                                                                                                                                                              output 1 consequents                                                                                                                                                                                         output 2 consequents                                                                                                                                                                                          output 3 consequents                                                                                                                                                                                         
      class             Alcohol   

## Evaluation

### Test data

In [97]:
pred = model.predict(x_test)

acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred, average='weighted', zero_division=0)
recall = recall_score(y_test, pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
#mul_label_conf_matrix = multilabel_confusion_matrix(y_test, pred)
class_rep = classification_report(y_test, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

#print("Multilabel Confusion Matrix:")
#print(mul_label_conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 0.9814814814814815
Precision: 0.9826388888888888
Recall: 0.9814814814814815
f1 score: 0.981554331672349 

Confusion Matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]] 

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54



### Training data

In [98]:
pred = model.predict(x_train)

acc = accuracy_score(y_train, pred)
prec = precision_score(y_train, pred, average='weighted', zero_division=0)
recall = recall_score(y_train, pred, average='weighted', zero_division=0)
f1 = f1_score(y_train, pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_train, pred)
class_rep = classification_report(y_train, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
f1 score: 1.0 

Confusion Matrix:
[[33  0  0]
 [ 0 40  0]
 [ 0  0 26]] 

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00        40
           2       1.00      1.00      1.00        26

    accuracy                           1.00        99
   macro avg       1.00      1.00      1.00        99
weighted avg       1.00      1.00      1.00        99



In [99]:
print(model.get_rules_structure().to_string())

        premises                                                                                                                                                                                                                                                                                                              output 1 consequents                                                                                                                                                                                         output 2 consequents                                                                                                                                                                                          output 3 consequents                                                                                                                                                                                         
           class             Alcohol           Malicacid         